<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-Archive-Downloader/blob/main/Colab-Archive-Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Internet Archive Downloader

Download books, movies, software from archive.org to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install internetarchive -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/InternetArchiveDownloader/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/InternetArchiveDownloader/'  # final destination

IDENTIFIER = ""  # Archive.org identifier
FILE_FILTER = ""  # Only download files containing this string

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, shutil
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def get_all_files(root):
    result = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            result.append(os.path.join(dirpath, f))
    return result

def main():
    from IPython.display import display, HTML, Javascript

    if not IDENTIFIER:
        print('ERROR: Set IDENTIFIER in config')
        return

    if KEEP_ALIVE:
        display(Javascript('''
            function keepAlive(){
                var btn=document.querySelector("colab-connect-button");
                if(btn)btn.click()
            }
            setInterval(keepAlive,120000);
        '''))
        print('Keep-alive active')

    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')

    begin = time.time()
    progress_display = display(HTML('<pre>Starting...</pre>'), display_id='dl-progress')

    from internetarchive import get_item
    item = get_item(IDENTIFIER)
    if not item.exists:
        print(f"Item not found: {IDENTIFIER}")
        return

    title = item.metadata.get('title', IDENTIFIER)
    print(f'Item: {title}')

    files_list = list(item.get_files())
    if FILE_FILTER:
        files_list = [f for f in files_list if FILE_FILTER.lower() in f.name.lower()]
        print(f'Filtered to {len(files_list)} files matching "{FILE_FILTER}"')
    print(f'Downloading {len(files_list)} files...')

    for i, f in enumerate(files_list):
        progress_display.update(HTML(f'<pre>Downloading ({i+1}/{len(files_list)}): {f.name} ({format_bytes(f.size)})</pre>'))
        f.download(SAVE_PATH)

    end = time.time()
    elapsed = int(end - begin)
    print()
    print('=' * 50)
    print('COMPLETE')
    print(f'Elapsed: {elapsed // 60}m {elapsed % 60}s')
    print(f'Saved locally: {SAVE_PATH}')

    items = get_all_files(SAVE_PATH)
    total_sz = sum(os.path.getsize(f) for f in items)
    print(f'Downloaded {len(items)} files ({format_bytes(total_sz)})')

    if len(items) > 1:
        import zipfile
        processed = 0
        zpath = SAVE_PATH.rstrip('/').rstrip('\\') + '.zip'
        print(f'\nZipping {len(items)} files...')
        with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fp in items:
                arcname = os.path.relpath(fp, SAVE_PATH)
                zf.write(fp, arcname)
                processed += os.path.getsize(fp)
                pct = int(processed * 100 / total_sz) if total_sz else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {arcname}</pre>'))
        zip_name = os.path.basename(zpath)
        zip_size = os.path.getsize(zpath)
        print(f'Zip: {zip_name} ({format_bytes(zip_size)})')

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for f in os.listdir(SAVE_PATH):
        shutil.move(os.path.join(SAVE_PATH, f), os.path.join(DRIVE_PATH, f))
    print(f'Final: {DRIVE_PATH}')

    if len(items) > 1:
        drive_zip = os.path.join(DRIVE_PATH, zip_name)
        if os.path.exists(drive_zip):
            display(HTML(f'<p>Zip saved to Drive:<br><a href="{drive_zip}" download>{zip_name} ({format_bytes(zip_size)})</a></p>'))
            if zip_size < 500 * 1024 * 1024:
                files.download(drive_zip)
            else:
                print(f'Large file ({format_bytes(zip_size)}) — open Drive to download.')

    print('=' * 50)

if __name__ == '__main__':
    main()
